In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("./uber.csv")
df.head()


In [ ]:
print(df.shape)
print(df.columns)
df.info()


In [ ]:
df['fare_amount'].isnull().sum()

In [ ]:
df =  df[(df['fare_amount']>=0) & (df['fare_amount']<=200)]
df =  df[(df['passenger_count']>=1) & (df['passenger_count']<=6)]


df = df[(df['pickup_longitude'] >= -74.25) & (df['pickup_longitude'] <= -73.7)]
df = df[(df['pickup_latitude'] >= 40.5) & (df['pickup_latitude'] <= 40.9)]
df = df[(df['dropoff_longitude'] >= -74.25) & (df['dropoff_longitude'] <= -73.7)]
df = df[(df['dropoff_latitude'] >= 40.5) & (df['dropoff_latitude'] <= 40.9)]


In [ ]:
df.shape

In [ ]:
def haversine(lat1,lon1,lat2,lon2):
  lat1,lon1,lat2,lon2=map(np.radians,[lat1,lon1,lat2,lon2])
  dlat=lat2-lat1
  dlon=lon2-lon1
  a=(np.sin(dlat/2)**2)+(np.cos(lat1)*np.cos(lat2)*(np.sin(dlon/2)**2))
  c=2*np.arcsin(np.sqrt(a))
  r=6371
  return c*r


In [ ]:
df['distance_km']=haversine(
    df['pickup_latitude'],df['pickup_longitude'],
    df['dropoff_latitude'],df['dropoff_longitude']
)

In [ ]:
df['pickup_datetime']=pd.to_datetime(df['pickup_datetime'])
df['hour']=df['pickup_datetime'].dt.hour
df['day_of_week']=df['pickup_datetime'].dt.dayofweek
df['month']=df['pickup_datetime'].dt.month
df['year']=df['pickup_datetime'].dt.year

In [ ]:
df=df[(df['distance_km']>0)]
df.shape


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.histplot(df['fare_amount'],bins=50,kde=True)
plt.title("Fare distribution")
plt.xlabel("Fare amt")
plt.ylabel("frequency")


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['distance_km'],df['fare_amount'])

In [ ]:
features = [
    'fare_amount', 'distance_km', 'passenger_count',
    'hour', 'day_of_week', 'month', 'year'
]
corr_matrix=df[features].corr()

sns.heatmap(corr_matrix, annot=True,cmap='coolwarm')
plt.figure(figsize=(10, 8))



In [ ]:
feat = ['distance_km', 'passenger_count', 'day_of_week', 'month', 'year']
X = df[feat]
y = df['fare_amount']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test samples:", X_test.shape[0])


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

In [ ]:
lr_r2 = r2_score(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
print(f"\nLinear Regression:")
print(f"  R-squared (R2): {lr_r2:.4f}")
print(f"  Root Mean Squared Error (RMSE): ${lr_rmse:.4f}")

In [ ]:
rf_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=10)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

In [ ]:
rf_r2 = r2_score(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
print(f"\nRandom Forest Regression:")
print(f"  R-squared (R2): {rf_r2:.4f}")
print(f"  Root Mean Squared Error (RMSE): ${rf_rmse:.4f}")